# VAE

Implementei um VAE para MNIST e comparei o que acontece ao reduzir o espaço latente de 16 para duas dimensões. Usei a mesma arquitetura, os mesmos dados e o mesmo orçamento de treinamento nos dois casos.

In [1]:
from pathlib import Path
import csv
import json
import platform
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import scipy
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms

if "__file__" in globals():
    BASE = Path(__file__).resolve().parents[1]
else:
    BASE = Path.cwd()
    if not (BASE / "code" / "vae.py").is_file():
        BASE = BASE / "docs" / "exercises" / "vae"
sys.path.insert(0, str(BASE / "code"))
from vae import load_data, VAE, loss_terms, run_epoch, evaluate, train_model

ROOT = BASE.parents[2]
FIGURES = BASE / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(2)
torch.use_deterministic_algorithms(True)
np.random.seed(42)
plt.rcParams.update({"font.size": 10, "figure.dpi": 120})


def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURES / filename, dpi=160)
    plt.close(fig)


def save_history(history, filename):
    with (BASE / "code" / filename).open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)

## Exercise 1

A VAE on MNIST.

### A — Load and split the data

Carreguei o MNIST com ToTensor, que deixa os pixels entre 0 e 1, e achatei cada imagem em 784 valores. O split usa exatamente random_split com gerador de seed 42: 50.000 imagens para treino e 10.000 para validação. As 10.000 imagens oficiais de teste ficam separadas. Os rótulos não entram no treinamento; servem para identificar os dígitos nas figuras.

Guardei os tensores após aplicar a transformação uma vez, para não repetir a conversão a cada época. O download fica em .cache/vae/data, fora dos arquivos versionados.

In [2]:
def load_data(data_dir):
    train_full = datasets.MNIST(
        data_dir, train=True, download=True, transform=transforms.ToTensor())
    test = datasets.MNIST(
        data_dir, train=False, download=True, transform=transforms.ToTensor())
    train, val = random_split(
        train_full, [50_000, 10_000],
        generator=torch.Generator().manual_seed(42))

    def flatten_once(dataset):
        # Aplico ToTensor uma vez e guardo os vetores para não repetir a
        # conversão de imagens PIL em todas as 20 épocas. A ordem é mantida.
        images, labels = [], []
        for x, y in DataLoader(dataset, batch_size=1024, shuffle=False):
            images.append(x.flatten(start_dim=1))
            labels.append(y)
        return TensorDataset(torch.cat(images), torch.cat(labels))

    return tuple(flatten_once(part) for part in (train, val, test))

train, val, test = load_data(ROOT / ".cache" / "vae" / "data")
print(f"Treino: {len(train)}; validação: {len(val)}; teste: {len(test)}")
print(f"Entrada: {train.tensors[0].shape}; pixels em "
      f"[{train.tensors[0].min().item()}, {train.tensors[0].max().item()}]")


  0%|          | 0.00/9.91M [00:00<?, ?B/s]


  0%|          | 32.8k/9.91M [00:00<00:32, 309kB/s]


  1%|          | 65.5k/9.91M [00:00<00:52, 188kB/s]


  1%|▏         | 131k/9.91M [00:00<00:32, 301kB/s] 


  3%|▎         | 262k/9.91M [00:00<00:18, 529kB/s]


  5%|▍         | 492k/9.91M [00:00<00:10, 912kB/s]

... (20 outputs left out)


100%|██████████| 1.65M/1.65M [00:01<00:00, 1.40MB/s]


  0%|          | 0.00/4.54k [00:00<?, ?B/s]


100%|██████████| 4.54k/4.54k [00:00<00:00, 7.04MB/s]

Treino: 50000; validação: 10000; teste: 10000
Entrada: torch.Size([50000, 784]); pixels em [0.0, 1.0]


### B — Build the model

O encoder tem as camadas 784 → 512 → 256, com ReLU após cada uma. A partir dos 256 valores, duas camadas lineares produzem μ e log σ², cada uma com 16 componentes. O decoder faz 16 → 256 → 512 → 784, com ReLU nas camadas ocultas e sigmoid na saída.

Escrevi a reparametrização como z = μ + exp(0,5 log σ²) · ε, com ε vindo de N(0, I). Assim, o sorteio fica em ε e o gradiente da reconstrução consegue chegar às duas saídas do encoder.

A perda é a soma da BCE de reconstrução com a KL, usando β = 1. A BCE soma os 784 pixels e a KL soma as dimensões latentes. Só depois divido pelo número de imagens do batch. A expressão fechada da KL é 0,5 ∑(μ² + exp(log σ²) − 1 − log σ²).

A sigmoid fornece uma probabilidade por pixel, então usei a BCE como perda de uma verossimilhança Bernoulli. Os pixels do MNIST são em grande parte próximos de zero ou um. Usar reduction="mean" na BCE também dividiria por 784 e mudaria a proporção entre os dois termos.

In [3]:
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU())
        self.mu = nn.Linear(256, latent_dim)
        self.logvar = nn.Linear(256, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 784), nn.Sigmoid())

    def encode(self, x):
        h = self.encoder(x)
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        # logvar = log(sigma²), então sigma = exp(logvar / 2).
        # O ruído independe dos pesos e o gradiente passa por mu e sigma.
        epsilon = torch.randn_like(mu)
        return mu + torch.exp(0.5 * logvar) * epsilon

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def loss_terms(reconstruction, x, mu, logvar):
    # Somo pixels na BCE e dimensões latentes na KL. Divido só pelo
    # número de imagens, mantendo os dois termos em nats por imagem.
    batch_size = x.shape[0]
    bce = F.binary_cross_entropy(reconstruction, x, reduction="sum") / batch_size
    kl = 0.5 * torch.sum(mu.square() + logvar.exp() - 1 - logvar) / batch_size
    return bce, kl

### C — Train

Treinei com Adam, taxa de aprendizado 0,001, batches de 128 e 20 épocas. Fixei torch.manual_seed(42) antes de construir o modelo e usei um gerador separado, também com seed 42, para embaralhar o treino. A execução usa CPU, duas threads e operações determinísticas.

Em cada época registrei reconstrução e KL separadamente. A média de treino considera todos os batches, ponderados pela quantidade de imagens. Avaliei a validação ao final de cada época, sem gradientes. Mantive o orçamento fixo e usei os pesos da época 20, sem parada antecipada ou escolha pelo teste.

Na validação e no teste, estimei a reconstrução com uma amostra de q(z|x) por imagem. Fixei a seed de avaliação em 123 dentro de fork_rng, para repetir o ruído sem mudar o gerador do treino. A KL é calculada pela fórmula fechada. Decodificar μ fica reservado às figuras de reconstrução e interpolação, pois não é a mesma estimativa da perda esperada.

In [4]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_bce, total_kl, n_images = 0.0, 0.0, 0
    with torch.set_grad_enabled(training):
        for x, _ in loader:
            if training:
                optimizer.zero_grad()
            reconstruction, mu, logvar = model(x)
            bce, kl = loss_terms(reconstruction, x, mu, logvar)
            if training:
                (bce + kl).backward()  # beta = 1 em todos os experimentos.
                optimizer.step()
            # Ponderar pelo tamanho evita dar peso excessivo ao último batch.
            n = x.shape[0]
            total_bce += bce.item() * n
            total_kl += kl.item() * n
            n_images += n
    return {"reconstruction": total_bce / n_images,
            "kl": total_kl / n_images,
            "negative_elbo": (total_bce + total_kl) / n_images}


def evaluate(model, dataset):
    # Uma amostra de q(z|x) por imagem estima a esperança da reconstrução.
    # Reutilizo o ruído na avaliação sem alterar o RNG do treinamento.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(123)
        loader = DataLoader(dataset, batch_size=128, shuffle=False)
        return run_epoch(model, loader)


def train_model(latent_dim, train, val):
    torch.manual_seed(42)  # Antes de construir CADA modelo.
    model = VAE(latent_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(train, batch_size=128, shuffle=True,
                        generator=torch.Generator().manual_seed(42))
    history = []
    for epoch in range(1, 21):
        train_metrics = run_epoch(model, loader, optimizer)
        val_metrics = evaluate(model, val)
        row = {"epoch": epoch}
        row.update({f"train_{k}": v for k, v in train_metrics.items()})
        row.update({f"val_{k}": v for k, v in val_metrics.items()})
        history.append(row)
        print(f"d={latent_dim:2d}, época {epoch:2d}: "
              f"treino BCE={train_metrics['reconstruction']:.3f}, "
              f"KL={train_metrics['kl']:.3f}; "
              f"val BCE={val_metrics['reconstruction']:.3f}, "
              f"KL={val_metrics['kl']:.3f}", flush=True)
    # Orçamento fixo: uso a época 20, sem escolher pelo teste.
    model.eval()
    return model, history

In [5]:
model16, history16 = train_model(16, train, val)
save_history(history16, "history-d16.csv")
test16 = evaluate(model16, test)
print("Teste d=16 (nats/imagem):", test16)

d=16, época  1: treino BCE=171.991, KL=7.540; val BCE=126.628, KL=13.013


d=16, época  2: treino BCE=114.584, KL=14.944; val BCE=106.132, KL=16.205


d=16, época  3: treino BCE=100.611, KL=17.257; val BCE=97.600, KL=17.785


d=16, época  4: treino BCE=94.680, KL=18.263; val BCE=93.469, KL=18.288


d=16, época  5: treino BCE=91.403, KL=18.753; val BCE=90.773, KL=18.600


d=16, época  6: treino BCE=89.183, KL=19.053; val BCE=88.378, KL=19.498


... (11 outputs left out)

d=16, época 18: treino BCE=80.302, KL=20.280; val BCE=81.385, KL=20.555


d=16, época 19: treino BCE=80.056, KL=20.304; val BCE=81.697, KL=20.143


d=16, época 20: treino BCE=79.733, KL=20.347; val BCE=81.840, KL=20.101


Teste d=16 (nats/imagem): {'reconstruction': 81.28727287597657, 'kl': 20.153741540527342, 'negative_elbo': 101.4410144165039}


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, term, title in zip(axes, ["reconstruction", "kl"], ["Reconstrução (BCE)", "KL"]):
    for split, label in [("train", "Treino"), ("val", "Validação")]:
        ax.plot(range(1, 21), [h[f"{split}_{term}"] for h in history16], label=label)
    ax.set(title=title, xlabel="Época", ylabel="Nats por imagem", xticks=[1, 5, 10, 15, 20])
    ax.legend()
fig.suptitle("Figura 1 — Termos da perda do VAE com d = 16")
save_figure(fig, "fig01-perdas.png")

No teste, a reconstrução foi 81,287, a KL foi 20,154 e a soma, o ELBO negativo estimado, foi 101,441 nats por imagem. A KL de validação passou de 13,013 na primeira época para 20,101 na última. O arquivo code/history-d16.csv guarda os dois termos de treino e validação nas 20 épocas.

![Figura 1 — Reconstrução e KL por época, para treino e validação.](figures/fig01-perdas.png)

Figura 1 — Os termos estão separados, em nats por imagem. A curva de treino é a média ponderada dos batches durante cada época, e a de validação usa os pesos ao final dela.

### D — Reconstructions and samples

Para cada dígito de 0 a 9, peguei sua primeira ocorrência no teste. Os índices, contando a partir de zero e nessa ordem, são 3, 2, 1, 18, 4, 8, 11, 0, 61 e 7. Na figura 2, passei cada imagem pelo encoder e decodifiquei μ, sem sortear z. Na figura 3, fiz o caminho oposto: sorteei 64 vetores do prior e passei diretamente pelo decoder.

In [7]:
x_test, y_test = test.tensors
indices = [int(torch.where(y_test == digit)[0][0]) for digit in range(10)]
originals = x_test[indices]
with torch.no_grad():
    mu, _ = model16.encode(originals)
    reconstructions = model16.decode(mu).reshape(-1, 28, 28)

fig, axes = plt.subplots(2, 10, figsize=(14, 3.7))
for col, (digit, index) in enumerate(zip(range(10), indices)):
    for row, image in enumerate([originals[col].reshape(28, 28), reconstructions[col]]):
        ax = axes[row, col]
        ax.imshow(image, cmap="gray", vmin=0, vmax=1)
        ax.set(xticks=[], yticks=[])
        if row == 0:
            ax.set_title(f"{digit} (índice {index})", fontsize=9)
        if row == 1:
            ax.set_xlabel("Coluna do pixel", fontsize=7)
    axes[0, col].set_ylabel("Linha do pixel", fontsize=7)
axes[0, 0].set_ylabel("Original\nLinha do pixel")
axes[1, 0].set_ylabel("Reconstrução\nLinha do pixel")
fig.suptitle("Figura 2 — Primeira imagem de cada dígito e reconstrução de μ (d = 16)")
save_figure(fig, "fig02-reconstrucoes.png")
print("Primeiros índices de teste por dígito 0–9:", indices)

Primeiros índices de teste por dígito 0–9: [3, 2, 1, 18, 4, 8, 11, 0, 61, 7]


![Figura 2 — Primeira imagem de cada dígito e reconstrução pela média latente.](figures/fig02-reconstrucoes.png)

Figura 2 — Originais na linha de cima e reconstruções de μ na linha de baixo, com d = 16.

In [8]:
torch.manual_seed(0)
with torch.no_grad():
    # Amostragem do prior escrita diretamente, sem helper de VAE.
    z = torch.randn(64, 16)
    samples = model16.decode(z).reshape(8, 8, 28, 28)
canvas = samples.permute(0, 2, 1, 3).reshape(8 * 28, 8 * 28)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(canvas, cmap="gray", vmin=0, vmax=1)
ax.set(title="Figura 3 — 64 amostras do prior N(0, I), d = 16, seed 0",
       xlabel="Coluna da amostra", ylabel="Linha da amostra",
       xticks=np.arange(8) * 28 + 13.5, xticklabels=np.arange(1, 9),
       yticks=np.arange(8) * 28 + 13.5, yticklabels=np.arange(1, 9))
save_figure(fig, "fig03-amostras.png")

![Figura 3 — Grade de 8 × 8 imagens geradas a partir do prior.](figures/fig03-amostras.png)

Figura 3 — Decodificação de 64 vetores de N(0, I), sorteados depois de torch.manual_seed(0).

### E — Analysis

Na figura 1, o modelo aceita pagar mais KL porque usar o código para guardar informação da imagem melhora bastante a reconstrução. Na validação, a BCE caiu de 126,628 para 81,840, enquanto a KL subiu de 13,013 para 20,101. A soma caiu de 139,641 para 101,941 nats por imagem. A redução da BCE compensa o aumento da KL. Para distinguir as imagens, o encoder afasta algumas médias de zero e ajusta as variâncias, pagando por se afastar do prior.

A figura 3 tem dígitos reconhecíveis, mas também formas ambíguas, como a imagem da linha 1, coluna 4. Uma razão está na localização dos códigos: a KL aproxima as distribuições do encoder de N(0, I), mas não faz a distribuição agregada dos códigos coincidir perfeitamente com o prior. Um sorteio pode cair numa região pouco visitada no treinamento ou entre regiões de dígitos diferentes. Na figura 2, o código vem de uma imagem real, então já aponta para uma região útil para reconstruí-la.

A outra razão está na saída do decoder para um z fixo. Aqui a sigmoid mostra probabilidades por pixel. Quando aquele código é compatível com traços diferentes, a BCE favorece probabilidades intermediárias nos pixels incertos. Ao exibir essas probabilidades como imagem, aparecem traços cinzentos e borrados. É a mesma ideia de combinar possibilidades discutida na aula de VAE, embora aqui a reconstrução use Bernoulli e BCE, em vez da hipótese Gaussiana com erro quadrático. Isso pode acontecer mesmo numa região latente bem treinada.

Usei μ na figura 2 para comparar cada original com uma reconstrução determinística. Sortear z acrescentaria variação da posterior e faria a mesma imagem produzir reconstruções diferentes a cada execução. μ representa o centro da distribuição codificada; decodificá-lo não equivale a calcular a média de todas as reconstruções, porque o decoder é não linear.

## Exercise 2

A Latent Space You Can See.

### A — Train a 2-D VAE

Treinei novamente mudando apenas a dimensão latente para 2. Mantive o split, a seed 42 antes da construção, Adam com taxa 0,001, β = 1, batch 128, as 20 épocas e o uso da última época. As cabeças e a entrada do decoder passam a ter duas componentes; o restante da arquitetura continua igual.

In [9]:
model2, history2 = train_model(2, train, val)
save_history(history2, "history-d2.csv")
test2 = evaluate(model2, test)
print("Teste d=2 (nats/imagem):", test2)
print("Comparação: d, reconstrução, KL, ELBO negativo")
for d, metrics in [(16, test16), (2, test2)]:
    print(d, *(round(metrics[k], 6) for k in ["reconstruction", "kl", "negative_elbo"]))

d= 2, época  1: treino BCE=181.244, KL=3.902; val BCE=159.123, KL=5.230


d= 2, época  2: treino BCE=156.016, KL=5.249; val BCE=152.872, KL=5.626


d= 2, época  3: treino BCE=150.734, KL=5.583; val BCE=148.647, KL=5.845


d= 2, época  4: treino BCE=147.385, KL=5.790; val BCE=146.909, KL=5.598


d= 2, época  5: treino BCE=144.924, KL=5.931; val BCE=144.203, KL=6.108


d= 2, época  6: treino BCE=143.167, KL=6.031; val BCE=142.882, KL=6.000


... (11 outputs left out)

d= 2, época 18: treino BCE=134.686, KL=6.638; val BCE=136.086, KL=6.488


d= 2, época 19: treino BCE=134.316, KL=6.654; val BCE=136.546, KL=6.629


d= 2, época 20: treino BCE=133.914, KL=6.693; val BCE=136.015, KL=6.547


Teste d=2 (nats/imagem): {'reconstruction': 136.0346826171875, 'kl': 6.540583154296875, 'negative_elbo': 142.57526577148437}
Comparação: d, reconstrução, KL, ELBO negativo
16 81.287273 20.153742 101.441014
2 136.034683 6.540583 142.575266


No modelo de duas dimensões, a reconstrução de teste foi 136,035 e a KL foi 6,541 nats por imagem. A comparação com o modelo anterior fica assim:

| Dimensão latente | Reconstrução de teste | KL de teste | ELBO negativo de teste |
|---|---|---|---|
| 16 | 81,287 | 20,154 | 101,441 |
| 2 | 136,035 | 6,541 | 142,575 |

Os valores estão em nats por imagem. Reduzir o código para duas dimensões aumentou a reconstrução em 54,747 nats e reduziu a KL em 13,613 nats. As curvas completas do segundo treinamento estão em code/history-d2.csv.

### B — The encoded test set

Passei as 10.000 imagens oficiais de teste pelo encoder com d = 2 e usei somente μ. Cada ponto da figura 4 corresponde a uma imagem, e a cor mostra o rótulo verdadeiro. Os rótulos não foram usados para aprender esse espaço.

In [10]:
with torch.no_grad():
    # Não há nova amostragem: o scatter usa a média codificada de cada imagem.
    encoded = torch.cat([model2.encode(x)[0] for x in x_test.split(128)]).numpy()
fig, ax = plt.subplots(figsize=(8.5, 7))
colors = plt.get_cmap("tab10")
for digit in range(10):
    points = encoded[y_test.numpy() == digit]
    ax.scatter(points[:, 0], points[:, 1], s=6, alpha=0.45,
               color=colors(digit), label=str(digit), rasterized=True)
ax.set(title="Figura 4 — Médias latentes das 10.000 imagens de teste (d = 2)",
       xlabel="μ₁", ylabel="μ₂", aspect="equal")
ax.legend(title="Dígito", bbox_to_anchor=(1.02, 1), loc="upper left", markerscale=2)
save_figure(fig, "fig04-espaco-latente.png")

![Figura 4 — Médias latentes das 10.000 imagens de teste, coloridas por dígito.](figures/fig04-espaco-latente.png)

Figura 4 — Espaço latente de duas dimensões, sem PCA ou outra projeção.

### C — The decoded plane

Usei 15 valores de q igualmente espaçados entre 0,05 e 0,95 e apliquei Φ⁻¹ em cada eixo. Isso leva a coordenadas entre −1,64485 e 1,64485. Mantive o maior z₂ na primeira linha, para comparar a posição das imagens com o scatter.

Os intervalos entre quantis consecutivos têm a mesma massa do prior em cada eixo e, como z₁ e z₂ são independentes, cada célula formada por esses intervalos também tem a mesma massa.

In [11]:
quantiles = np.linspace(0.05, 0.95, 15)
coordinates = norm.ppf(quantiles)
# A primeira linha é o maior z2: a posição vertical coincide com o scatter.
latent_grid = np.array([(z1, z2) for z2 in coordinates[::-1] for z1 in coordinates])
with torch.no_grad():
    decoded_grid = model2.decode(torch.tensor(latent_grid, dtype=torch.float32))
tiles = decoded_grid.reshape(15, 15, 28, 28).permute(0, 2, 1, 3).reshape(420, 420)
fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(tiles, cmap="gray", vmin=0, vmax=1)
positions = np.arange(15) * 28 + 13.5
ax.set(title="Figura 5 — Plano latente decodificado em 15 × 15 quantis (d = 2)",
       xlabel="z₁ = Φ⁻¹(q₁)", ylabel="z₂ = Φ⁻¹(q₂)",
       xticks=positions, xticklabels=[f"{z:.2f}" for z in coordinates],
       yticks=positions, yticklabels=[f"{z:.2f}" for z in coordinates[::-1]])
ax.tick_params(axis="x", labelrotation=90, labelsize=8)
ax.tick_params(axis="y", labelsize=8)
save_figure(fig, "fig05-plano-decodificado.png")

![Figura 5 — Plano latente decodificado numa grade de 15 × 15 quantis.](figures/fig05-plano-decodificado.png)

Figura 5 — Cada bloco mostra a saída do decoder para o par de coordenadas indicado nos eixos; o espaçamento uniforme na imagem representa quantis, e não distâncias iguais em z.

### D — Interpolation

Escolhi os índices 0 e 1 do teste oficial, contando a partir de zero: são os dígitos 7 e 2. Usei 10 valores de t, incluindo 0 e 1. Na primeira linha calculei (1 − t)xₐ + txᵦ diretamente nos pixels. Na segunda, decodifiquei (1 − t)μₐ + tμᵦ com o modelo de d = 16 do exercício 1. Os extremos da segunda linha são reconstruções, por isso não precisam ser idênticos aos originais.

In [12]:
index_a, index_b = 0, 1  # Primeiras duas imagens oficiais: dígitos 7 e 2.
t = torch.linspace(0, 1, 10).unsqueeze(1)
xa, xb = x_test[index_a:index_a + 1], x_test[index_b:index_b + 1]
pixel_interpolation = (1 - t) * xa + t * xb
with torch.no_grad():
    mua, _ = model16.encode(xa)
    mub, _ = model16.encode(xb)
    latent_interpolation = model16.decode((1 - t) * mua + t * mub)
fig, axes = plt.subplots(2, 10, figsize=(14, 3.7))
for row, images in enumerate([pixel_interpolation, latent_interpolation]):
    for col, image in enumerate(images):
        ax = axes[row, col]
        ax.imshow(image.reshape(28, 28), cmap="gray", vmin=0, vmax=1)
        ax.set(xticks=[], yticks=[])
        if row == 0:
            ax.set_title(f"t = {t[col].item():.2f}")
        else:
            ax.set_xlabel("Coluna do pixel", fontsize=7)
axes[0, 0].set_ylabel("Pixels\nLinha do pixel")
axes[1, 0].set_ylabel("Latente (d = 16)\nLinha do pixel")
fig.suptitle("Figura 6 — Interpolação do índice 0 (7) ao índice 1 (2)")
save_figure(fig, "fig06-interpolacao.png")
print(f"Interpolação: índice {index_a} (dígito {y_test[index_a].item()}) → "
      f"índice {index_b} (dígito {y_test[index_b].item()})")

Interpolação: índice 0 (dígito 7) → índice 1 (dígito 2)


![Figura 6 — Interpolação entre o índice 0, dígito 7, e o índice 1, dígito 2.](figures/fig06-interpolacao.png)

Figura 6 — Interpolação nos pixels na primeira linha e no espaço latente de 16 dimensões na segunda.

### E — Analysis

Na figura 4, os grupos de 4, 7 e 9 se sobrepõem na região à direita do centro, aproximadamente com μ₁ entre 0 e 1 e μ₂ entre −0,5 e 0,5. A figura 5 mostra uma transição parecida: na coluna z₁ ≈ 0,33, ao descer de z₂ ≈ 0,50 para −0,33, aparecem formas de 4, depois de 7 e depois de 9. No meio, alguns traços ficam abertos ou quase fechados, deixando a imagem ambígua. Também há mistura entre 5 e 8 perto da região superior central do scatter. Os grupos não viram classes perfeitamente separadas, até porque o modelo nunca recebeu os rótulos.

Com a arquitetura e a regularização usadas, duas componentes precisam resumir tanto a forma do dígito quanto as variações da escrita. O modelo acaba perdendo detalhes como espessura, inclinação, curvatura e fechamento dos traços. Na figura 5, regiões vizinhas repetem formas parecidas e, nas fronteiras, aparecem versões borradas ou intermediárias, em vez de preservar todas as maneiras de escrever um dígito. Isso combina com o aumento medido na reconstrução, de 81,287 para 136,035 nats por imagem. A KL menor, de 6,541 contra 20,154, não indica uma reconstrução melhor; o código paga menos KL e guarda menos informação útil neste treinamento.

Na figura 6, a linha latente mantém uma forma mais parecida com um único dígito ao longo da transição. O 7 vai curvando o traço de cima e formando a base do 2, embora os passos perto de t = 0,33 e 0,44 ainda sejam ambíguos. O decoder aprendeu a transformar códigos em formas de dígitos, e a regularização ajuda a tornar o caminho entre códigos mais contínuo. Isso favorece uma transformação plausível, sem garantir que todo ponto intermediário seja um dígito claro.

A linha dos pixels sobrepõe os dois desenhos: um vai apagando e o outro aparece no mesmo lugar. Perto de t = 0,44 e 0,56, vejo os traços do 7 e do 2 ao mesmo tempo, com intensidades menores. A média direta dos pixels não desloca ou reorganiza os traços para formar outro dígito, por isso deixa essa aparência de duas imagens misturadas.

In [13]:
results = {
    "test_d16": test16, "test_d2": test2,
    "validation_kl_d16_first": history16[0]["val_kl"],
    "validation_kl_d16_last": history16[-1]["val_kl"],
    "first_test_indices_by_digit": indices,
    "interpolation_indices": [index_a, index_b],
    "environment": {"python": platform.python_version(), "torch": torch.__version__,
                    "torchvision": torchvision.__version__, "numpy": np.__version__,
                    "scipy": scipy.__version__, "matplotlib": matplotlib.__version__,
                    "device": "cpu", "threads": torch.get_num_threads(),
                    "evaluation_seed": 123, "evaluation_samples_per_image": 1}}
(BASE / "code" / "results.json").write_text(json.dumps(results, indent=2) + "\n")
print(json.dumps(results, indent=2))

{
  "test_d16": {
    "reconstruction": 81.28727287597657,
    "kl": 20.153741540527342,
    "negative_elbo": 101.4410144165039
  },
  "test_d2": {
    "reconstruction": 136.0346826171875,
    "kl": 6.540583154296875,
    "negative_elbo": 142.57526577148437
  },
  "validation_kl_d16_first": 13.01283667602539,
  "validation_kl_d16_last": 20.101319903564452,
  "first_test_indices_by_digit": [
    3,
    2,
    1,
    18,
    4,
    8,
    11,
    0,
    61,
    7
  ],
  "interpolation_indices": [
    0,
    1
  ],
  "environment": {
    "python": "3.14.7",
    "torch": "2.15.0",
    "torchvision": "0.29.2",
    "numpy": "2.5.3",
    "scipy": "1.18.1",
    "matplotlib": "3.11.2",
    "device": "cpu",
    "threads": 2,
    "evaluation_seed": 123,
    "evaluation_samples_per_image": 1
  }
}


## Results summary

| # | Quantity | Value |
|---|---|---|
| 1 | Ex. 1 — test reconstruction, KL and negative ELBO (d = 16) | Reconstrução: 81,287; KL: 20,154; ELBO negativo: 101,441 nats/imagem |
| 2 | Ex. 1 — validation KL at the first and at the last epoch | Época 1: 13,013; época 20: 20,101 nats/imagem |
| 3 | Ex. 2 — test reconstruction and KL (d = 2) | Reconstrução: 136,035; KL: 6,541 nats/imagem |
| 4 | Ex. 2 — the two test indices interpolated | 0 (dígito 7) e 1 (dígito 2), índices a partir de zero |